<a href="https://colab.research.google.com/github/devpatel0005/Stock-Sentiment-Analysis-based-on-News/blob/main/Stock_Sentiment_Analysis_Using_LSTM_based_on_News.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
!pip install tensorflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.6/572.6 MB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.5/57.5 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 111.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 90.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.5/24.5 MB 104.0 MB/s eta 0:00:00
  Attempting uninstall: h5py
    Found existing installation: h5py 3.15.1
    Uninstalling h5py-3.15.1:
      Successfully uninstalled h5py-3.15.1
  Attempting uninstall: keras
    Found existing installation: keras 3.10.0
    Uninstalling keras-3.10.0:
      Successfully uninstalled keras-3.10.0


In [16]:
import tensorflow
from tensorflow import keras
from tensorflow.keras.models import Sequential
from keras.optimizers import Adam
from tensorflow.keras.layers import Dense,Embedding,Dropout,BatchNormalization,Bidirectional,LSTM
from keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import one_hot
import nltk
from nltk.stem import PorterStemmer
from nltk.corpus import stopwords
import matplotlib.pyplot as plt

In [17]:
nltk.download('punkt_tab')
nltk.download('stopwords')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

- The data set in consideration is a combination of the world news and stock price shifts available on Kaggle.
- There are 25 columns of top news headlines for each day in the data frame.
Data ranges from 2008 to 2016 and the data from 2000 to 2008 was scrapped from Yahoo finance.
- Labels are based on the Dow Jones Industrial Average stock index.
- Class 1 → the stock price increased.
- Class 0 → the stock price stayed the same or decreased.

In [23]:
import pandas as pd
df=pd.read_csv('/content/Data.csv',encoding='ISO-8859-1')

df.sample(5)

,Date,Label,Top1,Top2,Top3,Top4,Top5,Top6,Top7,Top8,...,Top16,Top17,Top18,Top19,Top20,Top21,Top22,Top23,Top24,Top25
3258,2013-02-28,0,'Marijuana cannon' used to fire drugs over US ...,South Africa shock as 'police dragging' video ...,The Pilot of the Hot Air Balloon that crashed ...,Australian Muslim men charged with assault for...,North Korean prison camp was the only life he ...,Secret War on Enemy Within: British terror sus...,Two to tango: China accuses U.S. of almost 100...,What Happened To The Aid Meant To Rebuild Haiti?,...,European Union Moves Toward Bonus Cap for Bankers,South African police drag man 400m behind poli...,Police assaulting a cameraman during a protest...,Shell says it will pause drilling in Arctic Oc...,Millionaire Dennis Tito to send couple on mann...,Australia makes largest meths seizure. 'Austra...,"First Tweet from North Korea - ""Hello World""",emember the hibernating Swede - Lived 2 months...,BBC News - Greek ex-mayor jailed for life for ...,FOSS Patents: UK judge who issued extreme ruli...
2495,2010-02-17,1,b'Dude arrested for lighting a cigarette on th...,b'The community of ultra-Orthodox Jews in Isra...,"b'Hired and armed by the CIA in the 1960s, the...",b'What could have been another tragedy at Vanc...,"b""IDF bans 45,000 Palestinians from using majo...",b'A N Korean woman\'s account of what life is ...,b'INTERPOL has issued Red Notices for 11 inter...,b'George Will Mocks GOP Obsession With Sarah P...,...,b'Plane crashes into Northwest Austin office b...,b'BT cotton is causing suicides to skyrocket i...,"b""Government Trojans, Orwellian Doublespeak an...",b'More than two dozen words are banned for non...,b'Saudi royal held over aides murder in London...,"b'Haiti ""restavek"" tradition called child slav...","b""'Dubai police chief 99% sure Israel behind H...",b'UK calls in Israeli ambassador as Dubai kill...,b'China has much to lose from Iran sanctions',"b'We need judges to investigate our spies, not..."
3969,2015-12-23,1,New law in India would try teens as adults for...,Colombia legalizes medical marijuana,Anonymous has declared cyber-war on Turkey: ''...,"Research reveals five major banks, JP Morgan, ...",Terror attack foiled in French region of Orleans,New Zealand Judge rules Kim Dotcom can be extr...,German court ruling: you can force your ex lov...,The Pirate Bay Co-Founder Makes Kopimashin Mus...,...,Air pollution in India now worse than China,Saudi Arabia 'jails reformist writer Zuhair Ku...,Ginger extremist who fantasized about killing ...,Germany withdraws Patriot missile defense syst...,Russia's lower house of parliament has approve...,Hamas leader expelled from Turkey,Twenty-three companies fined for causing fores...,Miss Iraq pageant held for first time in 43 years,Sex tape row: German court orders man to destr...,Brazilian president rejects settler leader as ...
2858,2011-07-26,0,"Internet users ""have a reasonable expectation ...",A German tourist is being hailed as a hero for...,Glenn Beck compares Norway's dead teenagers to...,"Today I'm proud to be Norwegian: ""Youth partie...","Israel has erupted in protest -- yes, you read...",Glenn Beck slammed by the Norwegian government...,BP reports quarterly profit of $5.3BILLION!,"150,000 people are gathered in Oslo right now,...",...,Neglecting The Lithuanian Holocaust --- \nHolo...,British Blogger Says He Has Recording That Pro...,Gulf oil spill victims weary of wait for payou...,The former Director of Public Prosecutions (DP...,Norway: Police ponder new Anders Behring Breiv...,Hamas hangs 2 convicted of spying for Israel:\...,"Anders Behring Breivik appears insane, says hi...",The victims of the Oslo bombing and Utya massa...,China has ordered a 2-month nationwide safety ...,Plane crash kills 78 in Morocco - CNN
3168,2012-10-16,1,British computer hacker Gary McKinnon will not...,"For eight months, starting last November, he w...",\nStarbucks paid no tax on UK earnings in the ...,Indian baby suffers horrendous burns in shocki...,The J

- ISO-8859-1, also known as Latin-1, is a character encoding standard that maps 256 different characters to numerical values. It's one of the most common encodings used for text in Western European languages

In [24]:
df.shape

(3992, 27)

# Splitting the Data into train and test

In [25]:
df['Date']

,Date
0,2000-01-03
1,2000-01-04
2,2000-01-05
3,2000-01-06
4,2000-01-07
...,...
3987,2016-01-21
3988,2016-01-22
3989,2016-01-25
3990,2016-01-26


In [26]:
train=df[df['Date']<'20150101']
test=df[df['Date']>'20141231']

In [27]:
test.shape

(269, 27)

In [28]:
train.shape

(3975, 27)

# Feature Engineering


In [29]:
data=train.iloc[:,2:27]
data.replace('[^a-zA-Z]',' ',regex=True,inplace=True) #replace the full stop punctuattion with space

#renaming the columns for ease of acess
new_index=[str(i) for i in range(25)] #since our data has 25 columns top 25 news headlines for stock
data.columns=new_index

In [30]:
data

,0,1,2,3,4,5,6,7,8,9,...,15,16,17,18,19,20,21,22,23,24
0,A hindrance to operations extracts from the...,Scorecard,Hughes instant hit buoys Blues,Jack gets his skates on at ice cold Alex,Chaos as Maracana builds up for United,Depleted Leicester prevail as Elliott spoils E...,Hungry Spurs sense rich pickings,Gunners so wide of an easy target,Derby raise a glass to Strupar s debut double,Southgate strikes Leeds pay the penalty,...,Flintoff injury piles on woe for England,Hunters threaten Jospin with new battle of the...,Kohl s successor drawn into scandal,The difference between men and women,Sara Denver nurse turned solicitor,Diana s landmine crusade put Tories in a panic,Yeltsin s resignation caught opposition flat f...,Russian roulette,Sold out,Recovering a title
1,Scorecard,The best lake scene,Leader German sleaze inquiry,Cheerio boyo,The main recommendations,Has Cubie killed fees,Has Cubie killed fees,Has Cubie killed fees,Hopkins furious at Foster s lack of Hannibal...,Has Cubie killed fees,...,On the critical list,The timing of their lives,Dear doctor,Irish court halts IRA man s extradition to Nor...,Burundi peace initiative fades after rebels re...,PE points the way forward to the ECB,Campaigners keep up pressure on Nazi war crime...,Jane Ratcliffe,Yet more things you wouldn t know without the ...,Millennium bug fails to bite
2,Coventry caught on counter by Flo,United s rivals on the road to Rio,Thatcher issues defence before trial by video,Police help Smith lay down the law at Everton,Tale of Trautmann bears two more retellings,England on the rack,Pakistan retaliate with call for video of Walsh,Cullinan continues his Cape monopoly,McGrath puts India out of their misery,Blair Witch bandwagon rolls on,...,South Melbourne Australia,Necaxa Mexico,Real Madrid Spain,Raja Casablanca Morocco,Corinthians Brazil,Tony s pet project,Al Nassr Saudi Arabia,Ideal Holmes show,Pinochet leaves hospital after tests,Useful links
3,Pilgrim knows how to progress,Thatcher facing ban,McIlroy calls for Irish fighting spirit,Leicester bin stadium blueprint,United braced for Mexican wave,Auntie back in fashion even if the dress look...,Shoaib appeal goes to the top,Hussain hurt by shambles but lays blame on e...,England s decade of disasters,Revenge is sweet for jubilant Cronje,...,Putin admits Yeltsin quit to give him a head s...,BBC worst hit as digital TV begins to bite,How much can you pay for,Christmas glitches,Upending a table Chopping a line and Scoring ...,Scientific evidence unreliable defence claims,Fusco wins judicial review in extradition case,Rebels thwart Russian advance,Blair orders shake up of failing NHS,Lessons of law s hard heart
4,Hitches and Horlocks,Beckham off but United survive,Breast cancer screening,Alan Parker,Guardian readers are you all whingers,Hollywood Beyond,Ashes and diamonds,Whingers a formidable minority,Alan Parker part two,Thuggery Toxins and Ties,...,Most everywhere UDIs,Most wanted Chloe lunettes,Return of the cane completely off the agenda,From Sleepy Hollow to Greeneland,Blunkett outlines vision for over s,Embattled Dobson attacks play now pay later ...,Doom and the Dome,What is the north south divide,Aitken released from jail,Gone aloft
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3970,Brazil declares emergency after babies a...,Scientists find big yr old Viking settlem...,Paris attacks Belgian police arrest ninth sus...,Wiretapping reveals communication between Turk...,Russia airstrikes Hundreds of Syrian civilans...,Pakistan changes neutral position on Syrian Ci...,North Korean diplomat arrested in South Africa...,German teachers want Mein Kampf on syllabus,North Korea slave force earns Kim Jong Un regi...,Yemeni Forces Preparing to Attack Saudi...,...,Report on sexual exploitation and abuse by pea...,Baby girl dies after X president s son s staff...,China smog sparks red alerts in cities,US planned East Berlin s systematic destructi...,Qatar World Cup workers e

# Converting the headlines into lowercase

In [31]:
for index in new_index:
  data[index]=data[index].str.lower()

In [32]:
data.head()

,0,1,2,3,4,5,6,7,8,9,...,15,16,17,18,19,20,21,22,23,24
0,a hindrance to operations extracts from the...,scorecard,hughes instant hit buoys blues,jack gets his skates on at ice cold alex,chaos as maracana builds up for united,depleted leicester prevail as elliott spoils e...,hungry spurs sense rich pickings,gunners so wide of an easy target,derby raise a glass to strupar s debut double,southgate strikes leeds pay the penalty,...,flintoff injury piles on woe for england,hunters threaten jospin with new battle of the...,kohl s successor drawn into scandal,the difference between men and women,sara denver nurse turned solicitor,diana s landmine crusade put tories in a panic,yeltsin s resignation caught opposition flat f...,russian roulette,sold out,recovering a title
1,scorecard,the best lake scene,leader german sleaze inquiry,cheerio boyo,the main recommendations,has cubie killed fees,has cubie killed fees,has cubie killed fees,hopkins furious at foster s lack of hannibal...,has cubie killed fees,...,on the critical list,the timing of their lives,dear doctor,irish court halts ira man s extradition to nor...,burundi peace initiative fades after rebels re...,pe points the way forward to the ecb,campaigners keep up pressure on nazi war crime...,jane ratcliffe,yet more things you wouldn t know without the ...,millennium bug fails to bite
2,coventry caught on counter by flo,united s rivals on the road to rio,thatcher issues defence before trial by video,police help smith lay down the law at everton,tale of trautmann bears two more retellings,england on the rack,pakistan retaliate with call for video of walsh,cullinan continues his cape monopoly,mcgrath puts india out of their misery,blair witch bandwagon rolls on,...,south melbourne australia,necaxa mexico,real madrid spain,raja casablanca morocco,corinthians brazil,tony s pet project,al nassr saudi arabia,ideal holmes show,pinochet leaves hospital after tests,useful links
3,pilgrim knows how to progress,thatcher facing ban,mcilroy calls for irish fighting spirit,leicester bin stadium blueprint,united braced for mexican wave,auntie back in fashion even if the dress look...,shoaib appeal goes to the top,hussain hurt by shambles but lays blame on e...,england s decade of disasters,revenge is sweet for jubilant cronje,...,putin admits yeltsin quit to give him a head s...,bbc worst hit as digital tv begins to bite,how much can you pay for,christmas glitches,upending a table chopping a line and scoring ...,scientific evidence unreliable defence claims,fusco wins judicial review in extradition case,rebels thwart russian advance,blair orders shake up of failing nhs,lessons of law s hard heart
4,hitches and horlocks,beckham off but united survive,breast cancer screening,alan parker,guardian readers are you all whingers,hollywood beyond,ashes and diamonds,whingers a formidable minority,alan parker part two,thuggery toxins and ties,...,most everywhere udis,most wanted chloe lunettes,return of the cane completely off the agenda,from sleepy hollow to greeneland,blunkett outlines vision for over s,embattled dobson attacks play now pay later ...,doom and the dome,what is the north south divide,aitken released from jail,gone aloft


- Now for implementing the NLP techniques like Bag of Words or Tf-IDF we need to combine these all columns of a row into one single paragraph for every row

In [33]:
data.index

Index([   0,    1,    2,    3,    4,    5,    6,    7,    8,    9,
       ...
       3965, 3966, 3967, 3968, 3969, 3970, 3971, 3972, 3973, 3974],
      dtype='int64', length=3975)

In [34]:
headlines=[]
for rows in range(0,len(data.index)):
  headlines.append(' '.join(str(x) for x in data.iloc[rows,0:25])) # columns are from 0 to 24 so we mentioned 0:25 to include all

In [35]:
headlines[1]

'scorecard the best lake scene leader  german sleaze inquiry cheerio  boyo the main recommendations has cubie killed fees  has cubie killed fees  has cubie killed fees  hopkins  furious  at foster s lack of hannibal appetite has cubie killed fees  a tale of two tails i say what i like and i like what i say elbows  eyes and nipples task force to assess risk of asteroid collision how i found myself at last on the critical list the timing of their lives dear doctor irish court halts ira man s extradition to northern ireland burundi peace initiative fades after rebels reject mandela as mediator pe points the way forward to the ecb campaigners keep up pressure on nazi war crimes suspect jane ratcliffe yet more things you wouldn t know without the movies millennium bug fails to bite'

- Now for every row we have merged  of each and every index the news into one single headline

In [36]:
ps=PorterStemmer()

In [37]:
stem_headlines=[]
for paragraph in headlines:
  processed_paragraph_words = []
  sentences=nltk.sent_tokenize(paragraph)
  for sentence in sentences:
    words_in_sentence = nltk.word_tokenize(sentence) # Tokenize the sentence into words
    for word in words_in_sentence:
      # Convert to lowercase before checking stopwords and stemming
      word_lower = word.lower()
      if word_lower not in stopwords.words('english'):
        processed_paragraph_words.append(ps.stem(word_lower))
  stem_headlines.append(' '.join(processed_paragraph_words)) # Join all processed words for the paragraph

In [38]:
all_words = set()
for headline in stem_headlines:
  for word in headline.split():
    all_words.add(word)
voc_size = len(all_words) + 1

print(f"Calculated vocabulary size: {voc_size}")

Calculated vocabulary size: 31838


In [39]:
onehot_repr = [one_hot(words, voc_size) for words in stem_headlines]
print(f"First one-hot encoded headline: {onehot_repr[0]}")

First one-hot encoded headline: [15884, 28878, 145, 4171, 7778, 31307, 25264, 17617, 833, 14184, 27549, 12061, 4197, 31504, 21052, 16370, 22887, 22106, 25901, 12337, 4700, 1666, 5308, 19699, 18610, 630, 31569, 29995, 18302, 19124, 7863, 13198, 5263, 1135, 16124, 28192, 22541, 12558, 13782, 16973, 19209, 391, 16494, 13173, 2181, 7991, 3415, 16633, 8542, 29244, 8021, 21564, 6817, 31687, 29995, 10570, 18918, 24805, 22901, 19417, 27993, 25052, 26929, 314, 4276, 9283, 6481, 19876, 833, 24286, 13073, 28693, 28166, 2980, 25732, 7872, 13917, 3854, 10852, 25778, 10418, 22133, 25428, 4061, 472, 27625, 30206, 2778, 16847, 22215, 22901, 29592, 21962, 15373, 27214, 16493, 921, 27514, 6981, 18761, 25299, 2049, 16282, 28321, 10419, 3651, 21743, 15970, 15559]


In [40]:
maxlen = max([len(x) for x in onehot_repr])
print(f"Maximum length of a headline for padding: {maxlen}")

Maximum length of a headline for padding: 479


In [41]:
embedded_docs = pad_sequences(onehot_repr, padding='pre', maxlen=maxlen)
print(f"Shape of embedded documents after padding: {embedded_docs.shape}")

Shape of embedded documents after padding: (3975, 479)


In [42]:
import numpy as np
y_train = np.array(train['Label'])

In [43]:
from sklearn.model_selection import train_test_split
x_train, x_val, y_train_split, y_val = train_test_split(embedded_docs, y_train, test_size=0.2, random_state=42)

print(f"Shape of x_train: {x_train.shape}")
print(f"Shape of x_val: {x_val.shape}")
print(f"Shape of y_train_split: {y_train_split.shape}")
print(f"Shape of y_val: {y_val.shape}")

Shape of x_train: (3180, 479)
Shape of x_val: (795, 479)
Shape of y_train_split: (3180,)
Shape of y_val: (795,)


In [44]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense,Embedding,Dropout,BatchNormalization,LSTM, Input, Reshape, Bidirectional # Added Bidirectional
from tensorflow.keras.regularizers import l2 # Import l2 regularizer


EMBEDDING_DIM = 100 # Define an appropriate embedding dimension

model=Sequential()
model.add(Embedding(input_dim=voc_size, output_dim=EMBEDDING_DIM))
model.add(Bidirectional(LSTM(128, activation='relu', return_sequences=True,kernel_regularizer=l2(0.01))))
model.add(Dropout(0.2))
model.add(Bidirectional(LSTM(64, activation='relu', return_sequences=True,kernel_regularizer=l2(0.01)))) # Keep return_sequences=True for stacking another BiLSTM
model.add(Dropout(0.2))
model.add(Bidirectional(LSTM(32, activation='relu',kernel_regularizer=l2(0.01))))
model.add(Dropout(0.2))
model.add(Dense(1,activation='sigmoid'))

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_2 (Bidirectional) │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [45]:
model.compile(optimizer=Adam(0.001),loss='binary_crossentropy',metrics=['accuracy'])

In [ ]:
history=model.fit(x_train,y_train,epochs=10,validation_split=0.1,batch_size=32)

Epoch 1/10
90/90 ━━━━━━━━━━━━━━━━━━━━ 106s 1s/step - accuracy: 0.5255 - loss: 4.5767 - val_accuracy: 0.5126 - val_loss: 1.3099
Epoch 2/10
45/90 ━━━━━━━━━━━━━━━━━━━━ 49s 1s/step - accuracy: 0.5085 - loss: 1.1371

In [ ]:
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])


In [ ]:
from sklearn.metrics import classification_report

# Make predictions on the training data
y_pred_train_proba = model.predict(x_train)
y_pred_train = (y_pred_train_proba > 0.5).astype(int)

# Generate classification report for training data
print("\nTraining Classification Report:")
print(classification_report(y_train_split, y_pred_train))

In [ ]:
from sklearn.metrics import classification_report

# Make predictions on the validation data
y_pred_val_proba = model.predict(x_val)
y_pred_val = (y_pred_val_proba > 0.5).astype(int)

# Generate classification report for validation data
print("\nValidation Classification Report:")
print(classification_report(y_val, y_pred_val))